# Column reference

**Current state** — `direction` is the open Supertrend flip and sets the structure:
**UP → bull put**, **DOWN → bear call**.

| Column | Meaning |
|---|---|
| `symbol` · `direction` | Ticker, and which way the open flip points |
| `trend_age_days` | Calendar days since the flip (weekends count) |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would invalidate the trend |
| `buffer_to_flip_pct` | Distance from spot to that level. **Small = about to invalidate** (NIFTY sat at 0.27% while topping the containment table) |
| `moved_since_entry_pct` | Signed *for* the position: positive = trend working, negative = underwater |
| `sector` · `beta` | From a hand-maintained map; beta vs NIFTY over ~250 days. Beta makes the 50/50 split *shock*-balanced, not just count-balanced |
| `realized_vol_pct` · `realized_vol_rank` | Annualized `window`-day realized vol, and where TODAY's value ranks (0-100) against this stock's own trailing ~2yr history. **Backfillable interim proxy for IV rank — not IV itself, no established threshold, informational only for now** (see Stage 8f) |

**Duration context** — descriptive only.

| Column | Meaning |
|---|---|
| `median_duration` | This stock's typical flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both tested whether crossing these
> predicts a reversal. **It does not.** `past_median = True` is not a warning and a
> negative runway is not an exit signal.

**Does the signal work here?** — all on closed flips in the *currently open* direction.

| Column | Meaning |
|---|---|
| `dir_flips` | Sample size; under 12 the edge columns return `None` |
| `dir_win_rate` | % of those flips that moved as the signal said |
| `dir_median_move` | Median favourable move. **Standalone for this side — not a comparison to the opposite side.** Negative = the typical flip goes the wrong way |

**Strike selection** — containment is the primary metric. A credit spread is a
"price won't breach my strike" bet, not a directional one.

| Column | Meaning |
|---|---|
| `contained_5pct` · `contained_6pct` | % of same-direction flips whose worst adverse excursion stayed inside that buffer |
| `worst_adverse` | Largest adverse excursion in 4yr — watch for fat tails behind good containment |
| `worst_gap` | Largest overnight gap against the position (open vs prev close) |
| `worst_day` | Largest full close-to-close day against it. Exists because `worst_gap` misses intraday collapses — PIIND's 8.97% fall was only a 1.28% gap |
| `pct_within_30d` | % resolved inside 30 days — matters for monthly expiries |
| `short_strike` · `strike_buffer_pct` | Strike implied by this stock's own containment, off live spot. Says nothing about whether the premium there is worth taking (needs IV — not built) |

**Was this flip normal?**

| Column | Meaning |
|---|---|
| `entry_day_move` | Signed move on the day the open flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%`. The containment stats above come from *ordinary* flips and do not describe a trend that began mid-crash |

**Wrong half of a good stock?**

| Column | Meaning |
|---|---|
| `opp_win_rate` · `opp_median_move` | The direction you are **not** trading. **Lower is not better** — these say nothing about the current trade. Their only job: current weak + opposite strong → watchlist; weak both ways → no edge |

**Verdicts**

| Column | Meaning |
|---|---|
| `edge_flag` | Empty = tradeable · `⚠ weak side` = watchlist · `⚠ NO EDGE EITHER SIDE` = fails both ways · `⚠ flip born on an outlier day` appended to any of these · `insufficient history` |
| `setup_score` | 0–100 composite (containment ×3, gap ×2, day ×2, win ×2, move ×1). **Relative to today's screen only** — not comparable across days |
| `reason` | Plain-language why, built from this stock's own numbers |

Bar for "has edge": `dir_win_rate >= 50` **and** `dir_median_move > 0`. Deliberately
plain, not optimised — nothing here is walk-forward validated. Names on the boundary
(BOSCHLTD `50.0/+0.08` passes, KOTAKBANK `50.0/−0.10` fails) are indistinguishable.

---

**Caveats**

1. **In-sample and descriptive** — same ~4 years the ranking is built from, no
   walk-forward validation. What the signal *has* done, not what it will do.
2. **Duration columns are pooled; edge columns are split by direction.**
3. **Containment spans the whole flip, not 30 days** — conservative.
4. **Not here at all:** earnings dates, real IV rank (a forward IV-history recorder started — `scripts/record_iv_snapshot.py` — but needs months to accumulate a usable percentile; `realized_vol_rank` above is the interim proxy), options liquidity, exits.


In [24]:
# Universe (2026-10-28, Stage 8i): NIFTY, BANKNIFTY, SENSEX + NIFTY 50 stocks
# that are gap-risk clean in the saved ranking (config/nifty50.csv). The other
# ~160 F&O stocks are no longer screened. No API calls here.


# autoreload: re-imports any changed src/*.py automatically, so an old kernel
# can never run stale module code. Without this, editing src/ and re-running
# without a kernel restart silently uses the OLD version — which is how 31
# reports once got rewritten missing their new columns.
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

from src.entry_advisor import stock_universe, INDEX_UNIVERSE

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")
stock_symbols = stock_universe()
final_candidates = credit_spread_ranking[
    credit_spread_ranking["symbol"].isin(stock_symbols + INDEX_UNIVERSE)].reset_index(drop=True)
# SENSEX may be missing from the ranking file; it is screened regardless
final_candidates = pd.concat([final_candidates, pd.DataFrame(
    {"symbol": [s for s in INDEX_UNIVERSE if s not in set(final_candidates["symbol"])]})],
    ignore_index=True)

print(f"{len(stock_symbols)} NIFTY 50 stocks + {', '.join(INDEX_UNIVERSE)}")
print("SORTED BY: credit_spread_score, descending (inherited from the ranking file).")
final_candidates

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
14 NIFTY 50 stocks + NIFTY, BANKNIFTY, SENSEX
SORTED BY: credit_spread_score, descending (inherited from the ranking file).


,symbol,credit_spread_score,gap_risk_flag,day_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr),Worst single-day move against (4yr),current_trend_age_days,trend_age_note
0,NESTLEIND,92.6,NaN,NaN,57.0,27.0,57.9,3.33,9.85,4.37,6.37,13.0,NaN
1,DRREDDY,85.3,NaN,⚠ HIGH 1-DAY MOVE RISK (>7.0%),62.0,23.1,66.1,3.29,7.95,4.96,8.94,16.0,NaN
2,ULTRACEMCO,80.4,NaN,NaN,70.0,20.1,65.7,3.09,11.62,3.61,6.86,12.0,NaN
3,BANKNIFTY,78.8,NaN,⚠ HIGH 1-DAY MOVE RISK (>7.0%),73.0,21.0,69.9,1.93,8.40,4.79,7.63,5.0,NaN
4,GRASIM,77.3,NaN,NaN,67.0,20.0,70.1,3.08,7.09,3.93,6.30,43.0,"note: aged trend — tested data shows momentum,..."
5,BAJAJFINSV,77.2,NaN,⚠ HIGH 1-DAY MOVE RISK (>7.0%),59.0,19.0,62.7,3.46,9.45,4.43,7.80,82.0,"note: aged trend — tested data shows momentum,..."
6,BHARTIARTL,74.4,NaN,⚠ HIGH 1-DAY MOVE RISK (>7.0%),64.0,21.0,68.8,3.18,13.86,4.25,7.72,22.0,NaN
7,ICICIBANK,74.1,NaN,NaN,72.0,21.0,73.6,2.84,8.86,4.97,6.52,22.0,NaN
8,TITAN,73.1,NaN,⚠ HIGH 1-DAY MOVE RISK (>7.0%),65.0,19.0,67.7,3.18,8.70,4.97,7.11,55.0,"note: aged trend — tested data shows momentum,..."
9,BAJAJ-AUTO,73.0,NaN,NaN,70.0,17.5,68.6,3.03,8.61,3.60,5.05,15.0,NaN


In [25]:
# For each of the ~50 symbols it looks up the right instrument 
# token (handling NIFTY/BANKNIFTY specially since indices live in a different token space), 
# fetches only the recent catch-up window of 4H candles from Kite (not the full 2022-to-now
# history - process_stock_incremental() merges this onto existing saved data instead of
# replacing it, same fix already applied to the debit spread/option buying/pledged pipelines,
# here for efficiency rather than a data-loss bug),
# reruns Supertrend → flip detection → monthly range → summary stats on the FULL merged
# history, and updates that symbol's _full_report.xlsx. This is the only block that talks to the API

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token, INDEX_SYMBOLS
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = INDEX_SYMBOLS   # SENSEX is on BSE

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/17] NESTLEIND — refreshed
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/DRREDDY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/DRREDDY_full_report.xlsx
[2/17] DRREDDY — refreshed
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ULTRACEMCO_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ULTRACEMCO_full_report.xlsx
[3/17] ULTRACEMCO — refreshed
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/BANKNIFTY_full_report.xlsx
[4/17] BANKNIFTY — refreshed

In [26]:
# reads the just-refreshed reports and reports current state:
# for each symbol, current trend direction, how many days the open trend has run,
# whether that age has crossed the stock's own median/75th/90th percentile duration,
# and runway_left_vs_median/p75/p90 - the same comparison as a signed number of
# days instead of three separate booleans (negative = already past that checkpoint) -
# sorted so the youngest (most recently flipped) trends show first. Same caveat as
# before: crossing a checkpoint is NOT a reversal/exit signal (tested, see README) -
# read runway as "how this compares to history," not as a warning.
#
# NEW - directional edge. The credit spread ranking read in cell 1 is built purely
# from risk-containment metrics (median/worst adverse, worst gap, duration, % within
# a monthly cycle) POOLED across both directions, and contains no profitability term
# at all. So a stock whose UP flips work and whose DOWN flips don't scores exactly
# like one where both work, and the shortlist gives no way to tell them apart. These
# columns answer, for the side that is actually open right now:
#   dir_win_rate/dir_median_move  - did the underlying actually move the way the
#       signal said? (directional read)
#   contained_5pct/contained_6pct - did price stay inside that buffer? This is what a
#       credit spread actually needs, since the structure is a "won't breach my
#       strike" bet, not a directional one. Read containment as PRIMARY and win rate
#       as a secondary tilt.
#   opp_win_rate/opp_median_move  - the other side, for contrast
# Caveat: in-sample and descriptive, same family as the runway columns - computed on
# the same ~4 years the ranking is built from, no walk-forward validation. It says
# what the signal HAS done here, not what it WILL do.

from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)

print("SORTED BY: trend_age_days, ascending — most recently flipped first.")
status.sort_values("trend_age_days")


SORTED BY: trend_age_days, ascending — most recently flipped first.


,symbol,direction,trend_age_days,flip_date,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,...,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move,edge_flag
10,APOLLOHOSP,UP,0,2026-10-28 09:15:00,20.0,False,False,False,20.0,35.0,...,82.4,10.47,4.34,5.79,61.8,3.67,False,42.1,-3.11,⚠ NO EDGE EITHER SIDE — signal has not worked ...
15,LT,UP,0,2026-10-28 09:15:00,20.4,False,False,False,20.4,28.6,...,94.3,7.70,3.40,12.46,71.4,3.39,False,47.4,-2.66,⚠ NO EDGE EITHER SIDE — signal has not worked ...
14,NIFTY,DOWN,5,2026-10-23 09:15:00,16.0,False,False,False,11.0,20.9,...,97.4,6.27,4.90,3.81,79.5,-0.39,False,50.8,-0.22,⚠ NO EDGE EITHER SIDE — signal has not worked ...
3,BANKNIFTY,DOWN,5,2026-10-23 09:15:00,21.0,False,False,False,16.0,28.0,...,94.4,8.40,4.79,5.79,72.2,-1.16,False,52.2,-0.17,⚠ NO EDGE EITHER SIDE — signal has not worked ...
13,SENSEX,DOWN,5,2026-10-23 09:15:00,17.0,False,False,False,12.0,21.6,...,100.0,5.95,4.52,3.99,79.5,-0.17,False,47.6,-0.32,⚠ NO EDGE EITHER SIDE — signal has not worked ...
16,EICHERMOT,DOWN,6,2026-10-22 09:15:00,18.2,False,False,False,12.2,27.2,...,61.8,10.93,4.97,8.95,82.4,-0.71,False,52.8,-1.77,⚠ NO EDGE EITHER SIDE — signal has not worked ...
11,HINDUNILVR,DOWN,6,2026-10-22 09:15:00,17.2,False,False,False,11.2,30.5,...,94.3,8.72,4.18,4.93,62.9,-1.49,False,42.3,-2.46,⚠ NO EDGE EITHER SIDE — signal has not worked ...
2,ULTRACEMCO,UP,12,2026-10-16 09:15:00,20.1,False,False,False,8.1,22.8,...,94.3,7.21,3.61,4.32,57.1,0.94,False,56.7,-1.50,⚠ NO EDGE EITHER SIDE — signal has not worked ...
12,JSWSTEEL,UP,12,2026-10-16 09:15:00,18.0,False,False,False,6.0,23.4,...,79.4,9.50,3.31,7.09,58.8,1.39,False,43.9,-2.88,⚠ NO EDGE EITHER SIDE — signal has not worked ...
0,NESTLEIND,DOWN,13,2026-10-15 13:15:00,27.0,False,False,False,14.0,31.0,...,92.9,9.85,4.15,5.32,71.4,-6.37,True,52.6,-1.04,⚠ NO EDGE EITHER SIDE — signal has not worked ...


In [27]:
# Splits the screen into the three buckets that call for DIFFERENT action, so a
# no-edge name can't sit next to a good one looking identical (which is exactly what
# the pooled ranking lets happen):
#
#   tradeable - the open direction has historically worked on this stock
#   watchlist - wrong side of a stock whose OTHER side works. Not a bad stock, just
#               the wrong half of it right now - revisit when it flips, don't discard
#   no_edge   - the signal has not worked in EITHER direction. This is the case the
#               ranking is blindest to: score only measures how well-behaved price
#               is, so a stock with tight, gap-free moves and a signal that's wrong
#               both ways scores well and looks like a good candidate. Waiting for a
#               flip does not fix these - the signal itself doesn't work here.
#
# Nothing is deleted - no_edge is returned, not dropped, so the exclusion stays
# visible and auditable. Same reasoning as the scorer's gap_risk_flag, which warns
# without removing the row.
#
# See the column reference at the top of this notebook for what each field means.

from src.daily_screener import split_by_edge

tradeable, watchlist, no_edge = split_by_edge(status)

edge_cols = ["symbol", "direction", "trend_age_days", "dir_flips", "dir_win_rate",
             "dir_median_move", "contained_5pct", "contained_6pct", "worst_adverse",
             "worst_gap", "worst_day", "pct_within_30d", "entry_day_move",
             "born_on_outlier_day", "opp_win_rate", "opp_median_move"]

print(f"=== NO EDGE EITHER SIDE ({len(no_edge)}) — do not trade, flipping won't help ===")
print("    SORTED BY: nothing — falls through in screen order (i.e. ranking order)")
display(no_edge[edge_cols])

print(f"\n=== WATCHLIST ({len(watchlist)}) — wrong side now, other side works ===")
print("    SORTED BY: opp_win_rate, descending — best flip candidates first")
display(watchlist[edge_cols].sort_values("opp_win_rate", ascending=False))

print(f"\n=== TRADEABLE ({len(tradeable)}) — open direction has worked historically ===")
print("    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the")
print("               primary metric, so a top win rate can still sit low in this table")
display(tradeable[edge_cols].sort_values(["contained_6pct", "dir_win_rate"], ascending=False))

# Crash-born flips, called out separately. These sit INSIDE the buckets above (a
# stock with real edge still lands in tradeable) but the containment stats there
# are drawn from ordinary flips and do not describe a trend that began mid-crash -
# PIIND topped this shortlist on 2026-08-12 on a flip created that morning by a
# -8.97% fall. Printed last so it's the thing left on screen before you act.
outliers = status[status["born_on_outlier_day"] == True]
if len(outliers):
    print(f"\n=== ⚠ FLIPS BORN ON AN OUTLIER DAY ({len(outliers)}) — stats above do not describe these ===")
    print("    SORTED BY: entry_day_move, ascending — biggest fall first")
    display(outliers[["symbol", "direction", "trend_age_days", "entry_day_move",
                      "dir_win_rate", "contained_6pct", "worst_gap", "worst_day"]]
            .sort_values("entry_day_move"))


=== NO EDGE EITHER SIDE (17) — do not trade, flipping won't help ===
    SORTED BY: nothing — falls through in screen order (i.e. ranking order)


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
0,NESTLEIND,DOWN,13,28,48.1,-1.88,78.6,92.9,9.85,4.15,5.32,71.4,-6.37,True,52.6,-1.04
1,DRREDDY,UP,16,31,56.5,-1.05,71.0,87.1,7.95,4.96,6.96,64.5,2.86,False,50.0,-1.18
2,ULTRACEMCO,UP,12,35,57.1,-1.02,85.7,94.3,7.21,3.61,4.32,57.1,0.94,False,56.7,-1.50
3,BANKNIFTY,DOWN,5,36,48.3,-0.62,94.4,94.4,8.40,4.79,5.79,72.2,-1.16,False,52.2,-0.17
4,GRASIM,DOWN,43,33,48.3,-2.86,78.8,93.9,7.09,3.93,5.55,78.8,-3.27,False,53.2,-1.38
5,BAJAJFINSV,DOWN,82,29,50.0,-2.75,69.0,79.3,9.45,4.43,7.80,58.6,-3.86,False,51.9,-1.48
6,BHARTIARTL,UP,22,32,56.2,-0.68,84.4,93.8,11.64,2.74,7.72,59.4,2.03,False,42.6,-2.92
7,ICICIBANK,UP,22,36,49.2,-0.97,94.4,97.2,8.86,2.98,6.52,72.2,0.40,False,52.4,-2.47
8,TITAN,DOWN,55,32,45.5,-1.50,81.2,90.6,6.23,4.57,6.74,78.1,-0.79,False,57.8,-1.77
9,BAJAJ-AUTO,UP,15,35,56.7,-0.98,82.9,88.6,7.65,3.22,4.93,65.7,3.26,False,43.6,-3.36



=== WATCHLIST (0) — wrong side now, other side works ===
    SORTED BY: opp_win_rate, descending — best flip candidates first


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move



=== TRADEABLE (0) — open direction has worked historically ===
    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the
               primary metric, so a top win rate can still sit low in this table


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move



=== ⚠ FLIPS BORN ON AN OUTLIER DAY (1) — stats above do not describe these ===
    SORTED BY: entry_day_move, ascending — biggest fall first


,symbol,direction,trend_age_days,entry_day_move,dir_win_rate,contained_6pct,worst_gap,worst_day
0,NESTLEIND,DOWN,13,-6.37,48.1,92.9,4.15,5.32


In [28]:
# ============================================================================
#  THE FINAL LIST — this is the only cell you need to read.
# ============================================================================
# Everything above is the working-out. This applies the whole filter chain and
# returns the book to trade, balanced 50/50 between bull puts and bear calls.
#
# Filters applied, in order (each one is reported below, nothing is silent):
#   1-2. directional edge    - SHOWN, NOT FILTERED since 2026-10-29: per-name
#                              win rates did not persist walk-forward (Stage 8o)
#   3. crash-born flip       - flip created by an outlier day; the containment
#                              stats come from ordinary flips and don't apply
#   4. thin buffer           - spot too close to the level that invalidates it
#   5. trend underwater      - thesis already failing before entry
#   6. weak trend at flip    - ADX(14) on the 4H flip candle not above 20
#                              (README Stage 8g; adopted 2026-10-28)
#   7. sector cap            - per DIRECTION (opposite bets on one sector hedge
#                              each other, they don't concentrate risk)
#   8. 50/50 direction       - so one market-wide shock can't hit everything
#
# Indices are a separate list (filters 1-5 only - no ADX, no sector cap, no
# 50/50; Stage 8i). Every pick is then priced as a real spread: listed strikes,
# hedge = skip one strike and buy the next, live credit, Kite margin, lots
# sized to the per-trade max-loss cap (src/spread_builder.py).
#
# Identical to `python scripts/daily_list.py`, which is the schedulable version.

import importlib
import src.daily_screener, src.market_context, src.entry_advisor, src.spread_builder
importlib.reload(src.daily_screener)
importlib.reload(src.market_context)
importlib.reload(src.entry_advisor)
importlib.reload(src.spread_builder)
from src.entry_advisor import build_entry_list, build_index_list
from src.spread_builder import add_spreads, risk_summary, CAPITAL, MAX_LOSS_PCT

index_final, index_excluded = build_index_list()
index_final = add_spreads(kite, index_final, set(INDEX_UNIVERSE))

final, excluded, summary = build_entry_list(stock_symbols, n_total=12, max_per_sector=2)
final = add_spreads(kite, final, set(INDEX_UNIVERSE))

SPREAD_COLS = ["symbol", "direction", "spot", "flip_level", "expiry", "short_strike", "long_strike",
               "strike_distance_pct", "credit", "quote_basis", "lots", "max_loss_total",
               "credit_total", "return_on_margin_pct", "strike_inside_sl", "spread_problem"]
risk = risk_summary(index_final)          # the traded book = indices (TRADING_RULES.md)
print(f"CAPITAL ₹{CAPITAL:,} · max loss per trade {MAX_LOSS_PCT}%")
if risk:
    print(f"BOOK if all taken: max loss ₹{risk['max_loss']:,} ({risk['max_loss_pct']}%) · "
          f"credit ₹{risk['credit']:,} · margin ₹{risk['margin']:,} ({risk['margin_pct']}%)")

from src.spread_builder import vix_context
vix = vix_context(kite)
print(f"India VIX {vix['vix']} — {vix['vix_pctile']}th percentile of the past year ({vix['vix_regime']})"
      + ("  ** high: index moves broke the implied range 2.4x as often (Stage 8j) **"
         if vix["vix_regime"] == "HIGH" else ""))
print("\nINDICES")
if index_final.empty:
    print("  none cleared the filters")
else:
    if index_final["direction"].nunique() == 1 and len(index_final) > 1:
        print("  ** all same direction — one correlated bet, consider taking only one **")
    display(index_final[[c for c in SPREAD_COLS + ["hist_strike_reach_pct"] if c in index_final]])
for key, names in index_excluded.items():
    if names:
        print(f"  {key:24s} {', '.join(names)}")

print("\nNIFTY 50 STOCKS — information only, not traded (Stage 8n)")

print(f"{summary['up']} BULL PUT  /  {summary['down']} BEAR CALL"
      f"      mean beta {summary['mean_beta_up']} UP vs {summary['mean_beta_down']} DOWN")
if summary["up"] != summary["down"]:
    print("  ** NOT BALANCED — too few DOWN-side candidates survived the filters.")
    print("     Trade fewer positions rather than forcing in a weak bear call.")
print("SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),")
print("           then setup_score descending WITHIN each direction block.")
print("           Which names got PICKED was decided by setup_score desc overall.")
print()

display(final[["symbol", "direction", "trend_age_days", "sector", "spot",
               "buffer_to_flip_pct", "dir_win_rate", "dir_median_move",
               "contained_6pct", "worst_gap", "worst_day", "beta", "adx_at_flip",
               "realized_vol_rank", "setup_score"]])
display(final[[c for c in SPREAD_COLS if c in final]])

# The WHY for each, so the list can be acted on without re-deriving anything.
for _, r in final.iterrows():
    print(f"\n{r['symbol']} — short strike {r['short_strike']:,.1f} "
          f"({r['strike_distance_pct']}% from spot)" if pd.notna(r.get("short_strike"))
          else f"\n{r['symbol']} — {r.get('spread_problem')}")
    print(f"   {r['reason']}")

print("\n" + "=" * 78)
print("EXCLUDED — considered and rejected")
for key, names in excluded.items():
    if names:
        print(f"  {key:24s} {', '.join(names)}")

print("\nSTILL MANUAL, not checked by any of the above:")
print("  earnings / corporate actions · IV rank · options liquidity · exits")
print("All statistics are in-sample and descriptive — what the signal HAS done,")
print("not what it will do.")


CAPITAL ₹1,000,000 · max loss per trade 2.0%
BOOK if all taken: max loss ₹90,120 (9.0%) · credit ₹11,880 · margin ₹458,424 (45.8%)
India VIX 16.25 — 76th percentile of the past year (HIGH)  ** high: index moves broke the implied range 2.4x as often (Stage 8j) **

INDICES
  ** all same direction — one correlated bet, consider taking only one **


,symbol,direction,spot,flip_level,expiry,short_strike,long_strike,strike_distance_pct,credit,quote_basis,lots,max_loss_total,credit_total,return_on_margin_pct,strike_inside_sl,hist_strike_reach_pct
0,BANKNIFTY,DOWN,54267.40,55534.12,2026-11-23,57100.0,57300.0,5.22,20.00,bid/ask,3,16200,1800,1.64,False,2.9
1,SENSEX,DOWN,73246.03,74898.33,2026-11-26,77800.0,78000.0,6.22,44.75,last traded (book empty),6,18630,5370,2.82,False,1.9


  trend_underwater         NIFTY

NIFTY 50 STOCKS
5 BULL PUT  /  1 BEAR CALL      mean beta 0.85 UP vs 1.32 DOWN
  ** NOT BALANCED — too few DOWN-side candidates survived the filters.
     Trade fewer positions rather than forcing in a weak bear call.
SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),
           then setup_score descending WITHIN each direction block.
           Which names got PICKED was decided by setup_score desc overall.



,symbol,direction,trend_age_days,sector,spot,buffer_to_flip_pct,dir_win_rate,dir_median_move,contained_6pct,worst_gap,worst_day,beta,adx_at_flip,realized_vol_rank,setup_score
4,BAJAJFINSV,DOWN,82,NBFC,1655.0,4.50,50.0,-2.75,79.3,4.43,7.80,1.32,39.2,39.2,13.3
2,ULTRACEMCO,UP,12,CEMENT,11180.0,4.35,57.1,-1.02,94.3,3.61,4.32,1.24,27.6,80.0,73.3
5,BHARTIARTL,UP,22,TELECOM,1955.2,4.17,56.2,-0.68,93.8,2.74,7.72,0.67,38.1,90.8,63.3
6,ICICIBANK,UP,22,BANK,1378.3,2.33,49.2,-0.97,97.2,2.98,6.52,0.91,30.1,66.2,61.7
8,BAJAJ-AUTO,UP,15,AUTO,10499.0,4.82,56.7,-0.98,88.6,3.22,4.93,0.91,51.3,96.0,61.7
9,APOLLOHOSP,UP,0,HEALTHCARE,8148.0,6.34,54.5,-2.04,82.4,4.34,5.79,0.54,35.7,100.0,36.7


,symbol,direction,spot,flip_level,expiry,short_strike,long_strike,strike_distance_pct,credit,quote_basis,lots,max_loss_total,credit_total,return_on_margin_pct,strike_inside_sl,spread_problem
4,BAJAJFINSV,DOWN,1655.0,1729.45,2026-11-23,1780.0,1820.0,7.55,5.05,bid/ask,1.0,10485.0,1515.0,5.75,False,NaN
2,ULTRACEMCO,UP,11180.0,10693.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no positive credit (bid/ask)
5,BHARTIARTL,UP,1955.2,1873.65,2026-11-23,1800.0,1760.0,7.94,2.55,bid/ask,1.0,17789.0,1211.0,2.58,False,NaN
6,ICICIBANK,UP,1378.3,1346.24,2026-11-23,1290.0,1270.0,6.41,1.35,bid/ask,1.0,13055.0,945.0,2.06,False,NaN
8,BAJAJ-AUTO,UP,10499.0,9993.33,2026-11-23,9700.0,9500.0,7.61,13.85,bid/ask,1.0,13961.0,1039.0,2.63,False,NaN
9,APOLLOHOSP,UP,8148.0,7631.24,2026-11-23,7500.0,7300.0,7.95,19.00,bid/ask,0.0,0.0,0.0,4.03,False,NaN



BAJAJFINSV — short strike 1,780.0 (7.55% from spot)
   BEAR CALL — trend is DOWN, sell calls above. Flip is 82d old vs a 19.0d median. DOWN trades here won 50.0% under the stop/expiry rule over 29 flips (does not predict the next one - Stage 8o); median flip-to-flip move -2.75%. 79.3% of them stayed inside 6% (worst 9.45%). Worst gap 4.43%, worst single day 7.8%. 58.6% resolved within 30d. Spot 1655.0 sits 4.5% from the 1729.45 flip level. Trend is working (+16.96% since flip). ADX 39.2 at the flip (> 20 required). Beta 1.32. Realized vol 20.6% (annualized), in its normal range vs its own 2yr history (39.2th pct — proxy only, not IV, no established threshold yet).

ULTRACEMCO — no positive credit (bid/ask)
   BULL PUT — trend is UP, sell puts below. Flip is 12d old vs a 20.1d median. UP trades here won 57.1% under the stop/expiry rule over 35 flips (does not predict the next one - Stage 8o); median flip-to-flip move -1.02%. 94.3% of them stayed inside 6% (worst 7.21%). Worst gap 3.61%

## Paper journal — every name on the final list, tracked to SL or expiry

Each symbol on the final list above is logged as a trade taken now (one open
trade per symbol — re-appearing tomorrow is the same trade). Every later run
checks all open trades, **whether or not they are still on the list**, for
exactly two exits:

- **SL_HIT** — a completed 4H candle closed beyond the trailing Supertrend level (the flip)
- **EXPIRED** — reached the current month's monthly expiry without an SL hit

`strike_touched` / `expired_beyond_strike` record whether the short strike was
reached — information, not an exit. Forward-only and append-only: a trade
exists because the list showed it that day, so it cannot be rebuilt later
(`logs/credit_spread_journal.csv`, tracked in git). See `src/credit_spread_journal.py`.

In [29]:
from src.credit_spread_journal import run_journal, summarize

# index trades only - stocks are information, not part of the traded system
journal, logged, problems = run_journal(kite, index_final, already_refreshed=symbols_to_refresh)
print(f"New trades logged this run: {', '.join(logged) if logged else 'none'}")
for p in problems:
    print(f"  ** {p}")

open_trades = journal[journal["status"] == "OPEN"]
print(f"\nOPEN — {len(open_trades)} trade(s), most recent first:")
display(open_trades.sort_values("logged_at", ascending=False)[
    ["symbol", "structure", "logged_at", "entry_spot", "short_strike", "long_strike", "initial_sl",
     "expiry", "lots", "max_loss_rupees", "max_adverse_pct", "strike_touched"]].reset_index(drop=True))

closed = journal[journal["status"] == "CLOSED"]
print(f"\nCLOSED — {len(closed)} trade(s):")
if not closed.empty:
    display(summarize(journal))
    display(closed.sort_values("exit_date", ascending=False)[
        ["symbol", "structure", "logged_at", "exit_date", "exit_reason", "entry_spot",
         "exit_spot", "sl_at_exit", "pct_move", "max_adverse_pct", "strike_touched",
         "expired_beyond_strike", "days_held"]].reset_index(drop=True))

Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/DABUR_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/DABUR_full_report.xlsx
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/HAVELLS_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/HAVELLS_full_report.xlsx
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/PIIND_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/PIIND_full_report.xlsx
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ICICIGI_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ICICIGI_full_report.xlsx
Fetching 2026-06-30 to 2026-10-29...
Saved 2378 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/TVSMOTOR_4hour.csv
Saved full

,symbol,structure,logged_at,entry_spot,short_strike,long_strike,initial_sl,expiry,lots,max_loss_rupees,max_adverse_pct,strike_touched
0,BAJAJFINSV,BEAR CALL,2026-10-29 04:13,1655.0,1780.0,1820.0,1729.45,2026-11-23,1.0,10485.0,NaN,False
1,BANKNIFTY,BEAR CALL,2026-10-28 19:52,54267.4,57100.0,57300.0,55534.12,2026-11-23,3,16200,0.0,False
2,SENSEX,BEAR CALL,2026-10-28 19:52,73246.03,74000.0,74200.0,74898.33,2026-10-29,5,16930,0.0,False
3,LICHSGFIN,BEAR CALL,2026-10-28 18:06,510.85,546.6,NaN,520.62,2026-11-23,NaN,NaN,0.0,False
4,POLYCAB,BEAR CALL,2026-10-28 18:06,7709.5,8249.2,NaN,7980.84,2026-11-23,NaN,NaN,0.0,False
5,BAJAJ-AUTO,BULL PUT,2026-10-28 18:06,10499.0,9764.1,NaN,9993.33,2026-11-23,NaN,NaN,0.0,False
6,APOLLOHOSP,BULL PUT,2026-10-28 18:06,8148.0,7659.1,NaN,7631.24,2026-11-23,NaN,NaN,0.0,False
7,ANANDRATHI,BULL PUT,2026-10-28 18:06,2271.0,2112.0,NaN,2194.53,2026-11-23,NaN,NaN,0.0,False
8,DABUR,BEAR CALL,2026-10-28 16:49,361.8,383.5,NaN,372.71,2026-11-23,NaN,NaN,0.0,False
9,HAVELLS,BEAR CALL,2026-10-28 16:49,1020.0,1081.2,NaN,1072.96,2026-11-23,NaN,NaN,0.0,False



CLOSED — 0 trade(s):
